# Flight Advection

This notebook demonstrates the core functionality of the `advection` library, which is used to model the movement (advection) of aircraft waypoints over time using weather data.

In [1]:
%load_ext autoreload
%autoreload 2

import datetime
from dateutil import tz
import sys
sys.path.append('../src')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pycontrails import Flight
import xarray as xr
import plotly.graph_objects as go

import advection.vectorized_prediction.advection as advection
import advection.utils.casting as casting
import advection.vectorized_prediction.time_util as time_util

## Define a Sample Flight

We define a synthetic flight segment representing an aircraft at altitude 10,668 m or 35,000 ft (~238 hPa) over the central US.

In [2]:
# Create a synthetic flight with multiple waypoints
df = pd.DataFrame({
    'time': [
        datetime.datetime(2023, 1, 1, 12, 0, 0, tzinfo=tz.UTC),
        datetime.datetime(2023, 1, 1, 12, 15, 0, tzinfo=tz.UTC),
        datetime.datetime(2023, 1, 1, 12, 30, 0, tzinfo=tz.UTC),
        datetime.datetime(2023, 1, 1, 12, 45, 0, tzinfo=tz.UTC),
        datetime.datetime(2023, 1, 1, 13, 0, 0, tzinfo=tz.UTC),
    ],
    'latitude': [40.0, 40.5, 41.0, 41.5, 42.0],
    'longitude': [-100.0, -99.0, -98.0, -97.0, -96.0],
    'altitude_ft': [35000.0, 35000.0, 36000.0, 36000.0, 37000.0],
    'flight_id': 'demo_flight'
})
fl = Flight(df)

# Convert pycontrails Flight to AdvectionState
# Here we explicitly convert the flight time to hours since epoch
time_hours = casting.timetype_to_float(fl.data['time'], unit=casting.Units.HOURS)
initial_state = advection.AdvectionState.from_flight(fl)

print(f"Created Flight with {len(fl)} waypoints.")
print(f"Initial State shape: {initial_state.pressure.shape}")


Created Flight with 5 waypoints.
Initial State shape: (5,)


## Setup Mock Weather Data

For this demo, we create a small synthetic `xarray` dataset containing wind velocities `u` (Eastward), `v` (Northward), and `w` (Vertical), and temperature `t` (K). In production, this would be loaded from regular netCDF or zarr files.

In [3]:
# Create a grid covering our flight path
lats = np.linspace(38, 44, 13)
lons = np.linspace(-102, -94, 17)
levels = np.array([200.0, 250.0, 300.0])
times = np.arange(time_hours.min() - 1, time_hours.max() + 3, 0.5)

# Create fluctuating wind blowing North-East
u_wind = (
    100.0
    + 0.5 * (levels - 250).reshape(1, -1, 1, 1)  # Vertical shear
    + 5.0 * np.sin(np.deg2rad(lats * 30)).reshape(1, 1, -1, 1) # Latitudinal variation
    + np.random.normal(scale=2.0, size=(len(times), len(levels), len(lats), len(lons))) # Noise
)
v_wind = (
    50.0
    - 0.2 * (lons + 98).reshape(1, 1, 1, -1)     # Longitudinal gradient
    + np.random.normal(scale=1.0, size=(len(times), len(levels), len(lats), len(lons))) # Noise
)
w_wind = np.random.normal(scale=0.01, size=(len(times), len(levels), len(lats), len(lons)))

# Standard temperature at flight levels (around 220 K)
t_temp = (
    220.0
    + 0.1 * (levels - 250).reshape(1, -1, 1, 1)  # Vertical lapse rate
    - 0.5 * (lats - 41).reshape(1, 1, -1, 1)    # Latitude gradient
    + 2.0 * np.sin(np.deg2rad(lons * 20)).reshape(1, 1, 1, -1) # Longitudinal wave
    + np.random.normal(scale=0.1, size=(len(times), len(levels), len(lats), len(lons))) # Noise
)

weather_ds = xr.Dataset(
    {
        "u": (["time", "level", "latitude", "longitude"], u_wind),
        "v": (["time", "level", "latitude", "longitude"], v_wind),
        "w": (["time", "level", "latitude", "longitude"], w_wind),
        "t": (["time", "level", "latitude", "longitude"], t_temp),
    },
    coords={
        "time": times,
        "level": levels,
        "latitude": lats,
        "longitude": lons,
    },
)
print("Synthetic weather bounds updated to cover flight path.")
print(
    "Latitudes:",
    weather_ds.latitude.values.min(),
    "to",
    weather_ds.latitude.values.max(),
)
print(
    "Longitudes:",
    weather_ds.longitude.values.min(),
    "to",
    weather_ds.longitude.values.max(),
)
print("Times:", weather_ds.time.values.min(), "to", weather_ds.time.values.max())


Synthetic weather bounds updated to cover flight path.
Latitudes: 38.0 to 44.0
Longitudes: -102.0 to -94.0
Times: 464603.0 to 464607.5


## Run Advection

We initialize the `LagrangianAdvector` with our weather data and advect the flight waypoint forward in time for 2 hours in 30-minute increments.

In [4]:
# Define evaluation times: each waypoint advected for 2 hours in 30-min steps
advection_length_hours = 2.0
step_size_secs = 1800.0

t_eval = time_util.t_eval_homogeneous(
    t_start=initial_state.time,
    advection_length_hours=advection_length_hours,
    step_size_secs=step_size_secs,
)

advector = advection.LagrangianAdvector(weather_ds)
# Advect with moist advection.
advected_flight = advector.advect(fl, t_eval, True)

print(
    f"Advected flight has {len(advected_flight)} waypoints (all steps combined)."
)

Advected flight has 13 waypoints (all steps combined).


/usr/local/google/home/jocelinl/git/google-contrails-attribution-reference/.venv/lib/python3.13/site-packages/pycontrails/core/flight.py:264: UserWarning: Sorting Flight data by time.
  warnings.warn("Sorting Flight data by time.")


## Visualize the Trajectory

Trace how the flight moved over the 2-hour advection period.

In [7]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

df = advected_flight.dataframe.copy()
# Ensure time is datetime
df['time'] = pd.to_datetime(df['time'])

fig = go.Figure()

# 1. Plot Original Path
fig.add_trace(go.Scatter3d(
    x=fl['longitude'],
    y=fl['latitude'],
    z=fl.level,
    mode='lines+markers',
    marker=dict(size=4, symbol='x', color='gray'),
    line=dict(dash='dash', color='gray'),
    name='Original Path'
))

# 2. Plot Full Advected Trajectories (the 'Movement')
cx, cy, cz = [], [], []
for pid, group in df.groupby('parcel_id'):
    group = group.sort_values('time')
    cx.extend(group['longitude'].tolist() + [None])
    cy.extend(group['latitude'].tolist() + [None])
    cz.extend(group['level'].tolist() + [None])

fig.add_trace(go.Scatter3d(
    x=cx, y=cy, z=cz,
    mode='lines',
    line=dict(color='red', width=2),
    opacity=0.4,
    name='Advection Trajectories'
))

# 3. Plot Time Increments (30-min steps) with Visible Labels
df['minutes'] = df.groupby('parcel_id')['time'].transform(lambda x: (x - x.min()).dt.total_seconds() / 60).astype(int)

fig.add_trace(go.Scatter3d(
    x=df['longitude'],
    y=df['latitude'],
    z=df['level'],
    mode='markers+text',
    marker=dict(size=3, color='red'),
    text=df['minutes'].apply(lambda x: f'{x}m'),
    textposition='top center',
    textfont=dict(size=10, color='red'),
    name='30-min Steps'
))

# 4. Highlight Final Positions
final_df = df.sort_values('time').groupby('parcel_id').tail(1)
fig.add_trace(go.Scatter3d(
    x=final_df['longitude'],
    y=final_df['latitude'],
    z=final_df['level'],
    mode='markers',
    marker=dict(size=6, color='blue'),
    name='Final Positions'
))

# 5. Add dotted line connecting final waypoints
fig.add_trace(go.Scatter3d(
    x=final_df['longitude'],
    y=final_df['latitude'],
    z=final_df['level'],
    mode='lines',
    line=dict(dash='dot', color='blue', width=2),
    name='Final Trajectory'
))

fig.update_layout(
    title="Multi-Waypoint Advection (Interactive 3D)",
    scene=dict(
        xaxis_title='Longitude',
        yaxis_title='Latitude',
        zaxis_title='Level (hPa)',
        zaxis=dict(autorange='reversed')
    ),
    width=1000, height=800,
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()
